# Data preparation and validation

## Research question

Are the current Dow 30 and synthetic inputs complete, correctly ordered, hash-verified, and prepared without test-period leakage?

In [ ]:
from pathlib import Path
from collections import deque
import json
import re
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (START, *START.parents) if (candidate / "pyproject.toml").is_file()),
    None,
)
assert ROOT is not None, "Could not locate the project root containing pyproject.toml."
print(f"Python: {sys.version.split()[0]}")
print(f"Project: {ROOT}")

## Inputs and provenance

The default path reads the current flat raw/interim/processed data layers and frozen inputs. Rebuilding market data is deliberately opt-in.

In [ ]:
REBUILD_MARKET_DATA = False
PREPARE_FROZEN_INPUTS = False
if REBUILD_MARKET_DATA:
    subprocess.run([sys.executable, "-m", "dtasrl.cli", "data", "build", "--config", "configs/data/dow30.yaml"], check=True, cwd=ROOT)
if PREPARE_FROZEN_INPUTS:
    subprocess.run([sys.executable, "-m", "dtasrl.cli", "data", "prepare"], check=True, cwd=ROOT)
print({"rebuild_market_data": REBUILD_MARKET_DATA, "prepare_frozen_inputs": PREPARE_FROZEN_INPUTS})

## Data checks

In [ ]:
from dtasrl.study_reports import data_report
report = data_report()
display(report["inputs"])
display(report["coverage"].head(30))
display(report["missing"])

## Configuration

In [ ]:
from dtasrl.config import load_config, public_config
cfg = public_config(load_config(ROOT / "configs/experiments/multi_asset.yaml"))
display(pd.DataFrame({"field": ["backend", "tradable tickers", "observed tickers", "cost bps", "steps per model"],
                      "value": [cfg["backend"], ", ".join(cfg["tickers"]), ", ".join(cfg["observation_tickers"]), cfg["transaction_cost_bps"], cfg["total_timesteps"]]}))

## Results

In [ ]:
meta = json.loads((ROOT / "outputs/00_data/source_metadata.json").read_text())
display(pd.DataFrame([{"first date": report["coverage"].first_date.min(),
                       "last date": report["coverage"].last_date.max(),
                       "tickers": report["coverage"].ticker.nunique(),
                       "processed rows": int(report["inputs"].query("layer == 'processed'").rows.iloc[0]),
                       "feature order": ", ".join(meta["feature_order"])}]))

## Figures

In [ ]:
for path in report["figures"]:
    display(Image(filename=str(path)))

## Interpretation and limitations

All 30 current constituents are backfilled to a common historical interval. This is appropriate for a controlled behavioral audit but creates survivorship bias and is not a historical investable Dow universe. Feature warm-up rows are excluded before fitting training-only scalers.

## Output inventory

In [ ]:
inventory = pd.read_csv(ROOT / "outputs/00_data/inventory.csv")
display(inventory)
print("All tables have CSV and Parquet copies; hashes are recorded in manifest.json.")

## Liquid 100 extension: optional data preparation

This independent extension freezes a current S&P 500 candidate list, corroborates it with IVV equity holdings, and selects 100 unique issuers using only pretraining history and 2009–2018 training information. It is a custom fixed stock universe, not the S&P 100 or a historical index reconstruction.

Set `PREPARE_LIQUID100 = True` below to download the candidate history, save the raw/interim/processed layers under `liquid100_` file prefixes, freeze selection and fit the training-only scaler. Existing completed extension data are verified and reused. None of the original Dow files or experiment outputs are replaced. No training is performed here. The current membership date is separate from the price cutoff of 2026-08-31.

In [ ]:
from pathlib import Path
import sys
START_LIQUID = Path.cwd().resolve()
ROOT_LIQUID = next(p for p in (START_LIQUID, *START_LIQUID.parents) if (p / "pyproject.toml").is_file())
if str(ROOT_LIQUID) not in sys.path:
    sys.path.insert(0, str(ROOT_LIQUID))
from scripts.liquid100_extension import prepare_data, data_report, DATA_REPORT

PREPARE_LIQUID100 = False
if PREPARE_LIQUID100:
    frozen_liquid100 = prepare_data()
    print(f"Frozen 100-stock input: {frozen_liquid100}")
else:
    print("Liquid 100 download/preparation is disabled; existing data are not changed.")

In [ ]:
import pandas as pd
from IPython.display import Image, display
if (ROOT_LIQUID / "data/processed/liquid100_metadata.json").is_file():
    liquid100_report = data_report()
    print("Membership crosscheck:", liquid100_report["metadata"]["candidate_crosscheck"])
    display(liquid100_report["selected"][["rank", "ticker", "company", "sector", "training_coverage", "median_training_dollar_volume"]])
    display(liquid100_report["audit"].groupby("reason").size().rename("stocks").reset_index())
    display(liquid100_report["coverage"].groupby("split")[["rows", "missing_features"]].agg(["min", "max"]))
    for path in liquid100_report["figures"]:
        display(Image(filename=str(path)))
    print("Full selection audit and data report:", DATA_REPORT.relative_to(ROOT_LIQUID))
else:
    print("No frozen Liquid 100 data yet. Enable PREPARE_LIQUID100 before running Notebook 05.")